# Primalidad

El objetivo es obtener un método eficiente para decidir si un entero es primo o no. (En realidad, *probablemente primo*.) 

Buscaremos condiciones necesarias para que un entero sea primo.

* <a href="#generadores" id="igeneradores">Generadores (raíces primitivas)</a>
* <a href="#pp" id="ipp">Pseudoprimos (pp)</a>
* <a href="#ppf" id="ippf">Pseudoprimos fuertes (ppf). Test de Miller Rabin</a>
* <a href="#test" id="itest">Test de primalidad</a>

---

### Primalidad versus factorización

In [1]:
from IPython.display import display, Latex

'''
Funciones auxiliares
'''

def primo_2q1(bits):
    '''
    Devuelve un primo p de la forma 2*q+1 siendo q primo también
    '''
    q = next_probable_prime(randint(2**(bits-1), 2**bits))
    p = 2*q+1

    while is_prime(p) == False:
        q = next_probable_prime(randint(2**(bits-1), 2**bits))
        p = 2*q + 1
    return p

def contar_bases_pp(n):
    b = 1
    pseudoSI = pseudoNO = 0

    while b<n:
        while(gcd(n, b) != 1):
            b = b+1
        aux = pow(b,n-1,n)
        if aux == 1:
            pseudoSI = pseudoSI+1
        else:
            pseudoNO = pseudoNO+1
        b = b+1
    show(html('{0} SÍ es pseudoprimo en {1} bases'.format(n,pseudoSI)))
    show(html('{0} NO es pseudoprimo en {1} bases'.format(n,pseudoNO)))


In [2]:
NumerodDeBits = 1024*4
for _ in range(5):
    r1 = randint(0, 2^(NumerodDeBits/2))
    r2 = randint(0, 2^(NumerodDeBits/2))
    print("Generar primos", NumerodDeBits/2, "bits")
    t0 = cputime()
    p1 = next_probable_prime(r1)
    print("Tiempo para generar p: ", cputime()-t0, flush=True)
    t0 = cputime()
    p2 = next_probable_prime(r2)
    print("Tiempo para generar q: ", cputime()-t0, flush=True)
    n = p1*p2
    r3 = randint(0, 2^NumerodDeBits)
    print("Generar primo", NumerodDeBits, "bits")
    t0 = cputime()
    p3 = next_probable_prime(r3)
    print("Tiempo para generar primo de", NumerodDeBits, "bits: ", cputime()-t0, flush=True)
    proof.arithmetic(False)
    print("Comprobar si compuesto/primo", NumerodDeBits, "bits")
    t0 = cputime()
    is_prime(n)
    print("Tiempo para entero compuesto de", NumerodDeBits, "bits: ", cputime()-t0, flush=True)
    t0 = cputime()
    is_prime(p3)
    print("Tiempo para primo de", NumerodDeBits, "bits: ", cputime()-t0, flush=True)
    print()


Generar primos 2048 bits
Tiempo para generar p:  1.2187900000000003
Tiempo para generar q:  0.603834
Generar primo 4096 bits
Tiempo para generar primo de 4096 bits:  13.115255
Comprobar si compuesto/primo 4096 bits
Tiempo para entero compuesto de 4096 bits:  0.020444999999998714
Tiempo para primo de 4096 bits:  0.058115000000000805

Generar primos 2048 bits
Tiempo para generar p:  0.20922900000000055
Tiempo para generar q:  0.4896649999999987
Generar primo 4096 bits
Tiempo para generar primo de 4096 bits:  7.6107890000000005
Comprobar si compuesto/primo 4096 bits
Tiempo para entero compuesto de 4096 bits:  0.019660999999999262
Tiempo para primo de 4096 bits:  0.06069900000000317

Generar primos 2048 bits
Tiempo para generar p:  2.7374299999999963
Tiempo para generar q:  0.291755000000002
Generar primo 4096 bits
Tiempo para generar primo de 4096 bits:  5.476505000000003
Comprobar si compuesto/primo 4096 bits
Tiempo para entero compuesto de 4096 bits:  0.01981500000000125
Tiempo para pri

### Tiempos de factorización
#### n=p*q
#### p,q~$2^{50} \text{ tarda }0,05s$
#### p,q~$2^{100} \text{ tarda }5 s$
#### p,q~$2^{110} \text{ tarda } 25s$ 
#### p,q~$2^{120} \text{ tarda }  75-115s$

In [3]:
bits = 100
limite = 2^bits
for _ in range(5):
    r1 = randint(0, limite)
    r2 = randint(0, limite)

    p1 = next_probable_prime(r1)
    p2 = next_probable_prime(r2)
    n = p1*p2

    t0 = cputime()
    aux = factor(n)
    print("Factorización:", aux)
    print("Tiempo=", cputime()-t0)
    print()

Factorización: 334206645390883873740724761217 * 1123355062968374386566663945163
Tiempo= 3.1832330000000013

Factorización: 163531487029850228103764332121 * 595741486182908911492110661823
Tiempo= 2.8618489999999994

Factorización: 775417609408445483952657882857 * 877607406742899754957909704889
Tiempo= 3.514510999999999

Factorización: 800623554551225130135639751367 * 1108947727092869643273793159159
Tiempo= 3.4107930000000053

Factorización: 82841500339106862547423179163 * 1077873722335544784323029920407
Tiempo= 3.0644769999999966



## <a href="#igeneradores" id="generadores"> Generadores (raíces primitivas)</a>

**Definición.**  Sea $a\in\mathbb Z_n^*$, diremos que $r$ es el 
orden de $a$ módulo $n$ si es el menor entero positivo tal que
$$
a^r\equiv 1\mod n
$$


**Obsevación:** $r\vert \varphi(n)$.

**Definición.**  $g\in\mathbb Z_n^*$ es `generador` de $Z_n^*$ si su orden es $\varphi(n)$.

**Observaciones:**
* Si $p$ es primo existe $g$, generador de $\mathbb Z_p^*=\{1,2,\ldots,p-1\}$, tal que  $$\mathbb Z_p^*=\{g^1,g^2,\ldots,\underbrace{g^{p-1}}_{=1}\}$$
* Hay $\varphi(p-1)$ generadores o raíces primitivas módulo $p$.
* $\mathbb Z_n^*$ tiene generadores si y solamente si $n=2, 4, p^k, 2p^k$, $p$ primo impar. El número de generadores es $\varphi(\varphi(n))$.
* Sea $g$, generador de $\mathbb Z_n^*$. Entonces $g^r$ es generador si y solamente si $\text{mcd}(r,\varphi(n))=1$.



>###  Ejemplo 
>
>$p=2q+1$, $p$, $q$ primos
>
> Hallemos el orden de distinto elementos


In [5]:
bits = 64

p = primo_2q1(bits)

q = Integer((p-1)/2)

ordenes = 1, 2, q, 2*q

print()
display(Latex('$p={}=2*{}+1$'.format(p, q)))
print()
display(Latex('Los órdenes posibles son: {}'.format(ordenes)))

for base in [2, 3, 4, 5, 8, p-1]:
    show(html('<br></br>Base: {}'.format(base)))
    for i in ordenes:
        aux=pow(base, i, p)
        display(Latex('${3}^{0}\\equiv {1} \\mod {2}$'.format(str({i}), aux, p, base)))


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<br></br>Base: 2

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<br></br>Base: 28213672854271378366

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

> ### Ejemplo
>
> Dado un primo $p$ hallemos un elemento de un orden primo determinado


In [4]:
bits = 8
p = next_probable_prime(randint(1, 2**bits))
#p=131

factores = factor(p-1)
divisores = divisors(p-1)

display(Latex('$p={0}$'.format(p)))
display(Latex('$p-1={0}={1}$'.format(p-1, factores)))
display(Latex('Los órdenes posibles son: ${0}$'.format(divisores)))


orden = [i for i in divisores if is_prime(i)]
orden = orden[-1]

e = Integer((p-1)/orden)

display(Latex('Buscaremos elementos de orden: ${0}$'.format(orden)))
display(Latex('Sea $e=\\frac{0}{1}={2}$'.format(str({p-1}), str({orden}),e)))

candidatos=[]
for base in range(1, p):
    aux=pow(base, e, p)
    if aux!=1:
        candidatos.append(aux)
    display(Latex('${3}^{0}\\equiv {1} \\mod {2}$'.format(str({e}), aux, p, base)))

display(Latex('Los elementos de orden primo {0} son: ${1}$'.format(orden, set(candidatos))))



<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

> ### Ejemplo
>
> Dado un primo $p$ hallemos un generador


In [8]:
bits = 32

p = next_probable_prime(randint(1, 2**bits))

p = primo_2q1(bits)

factores = factor(p-1)
divisores = divisors(p-1)

display(Latex('$p={0}$'.format(p)))
display(Latex('$p-1={0}={1}$'.format(p-1, factores)))
display(Latex('Los órdenes posibles son: ${0}$'.format(divisores)))


ordenes = divisores[:-1]
es_generador = False


while es_generador == False:
    es_generador = True
    base = randint(1, 2**bits)
    print('.'+str(base), end='.')
    for e in ordenes:
        aux = pow(base, e, p)
        if aux == 1:
            es_generador = False


display(Latex('Un generador es $g={0}$'.format(base)))

for e in divisores:
    aux = pow(base, e, p)
    display(Latex('${3}^{0}\\equiv {1} \\mod {2}$'.format(str({e}), aux, p, base)))


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

.993563689.

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<br>


**Definición.** Diremos que $a\in\mathbb Z_n^*$ es `residuo cuadrático` módulo $n$ si existe $b\in\mathbb Z_n^*$ tal que $a\equiv b^2\mod n$. O sea, si la ecuación $X^2\equiv a \mod n$ tiene solución en $Z_n^*$.

> ### Ejemplo
>
> Residuos cuadráticos módulo $p$
>

In [5]:
bits = 9
p = next_probable_prime(randint(2, 2**bits))

residuos = []
for k in range(2, p):
    residuos.append(pow(k, 2, p))
residuos = sorted(list(set(residuos)))

display(Latex('$p={0}$'.format(p)))

display(Latex('Los residuos cuadráticos son: ${0}$'.format(residuos)))

print('Número de residuos:', len(residuos))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Número de residuos: 83


**Teorema.** Sea $p$ primo impar y $g$ generador de $\mathbb Z_p^*$. Entonces:

<center>
$a\in \mathbb Z_p^*$ es residuo cuadrático módulo $p$ $\Leftrightarrow$ $a\equiv g^{2k}\mod p$.
</center>
 
 

Además, exactamente la mitad de los elementos de $\mathbb Z_p^*$ son residuos cuadráticos.

**Corolario.** 
$a\in \mathbb Z_p^*$ es residuo cuadrático módulo $p$ $\Leftrightarrow$ $a^{\frac{p-1}{2}}\equiv 1\mod p$.

**Corolario.** 
Sea $p$ primo impar y $a$ residuo cuadrático módulo $p$.
La ecuación $X^2\equiv a \mod p$ tiene exactamente dos soluciones: $g^k$ y $g^{k+\frac{p-1}{2}}$ (siendo $a\equiv g^{2k} \mod p$).

**Corolario.** 
Sea $p$ primo impar. 
La ecuación $X^2\equiv 1 \mod p$ tiene exactamente las soluciones $X \equiv \pm 1 \mod p$.






> ### Ejemplo
>
>$X^2\equiv 1 \mod 15=3\cdot5$, $\quad x\equiv \pm 1, \pm 4 \mod 15$
>
> $X^2\equiv 1 \mod 105=3\cdot5\cdot7$, $\quad x\equiv \pm 1, \pm 29, \pm 34, \pm 41 \mod 105$
>


## <a href="#ipp" id="pp"> 2. Pseudoprimos</a>

**Definición.** Diremos que un entero positivo $n$ pasa el `test de Fermat` en base $b$, $\text{mcd}(b,n)=1$, si $b^{n-1}\equiv 1 \mod n$.

**Observaciones:**
* Si $\text{mcd}(b,n)\neq 1$, $1<b<n$, entonces $n$ es compuesto. El coste de calcular $mcd(b,n)$ es cuadrático en el número de bits de $n$; el coste de calcular $b^{n-1} \mod n$ es cúbico en el número de bits de $n$; 
* Si $n$ es primo, pasa el test de Fermat en cualquier base.

**Definición.** Diremos que un entero compuesto $n$ es `pseudoprimo` en base $b$ si pasa el test de Fermat en base $b$.



> ### Ejemplo
> $n$ compuesto, pseudoprimo en diferentes bases $b$

In [10]:
bits_p = 4
bits = 2*bits_p
p = next_probable_prime(randint(1, 2**bits_p))
q = next_probable_prime(randint(1, 2**bits_p))
n = p*q
#n=91

show(html('$n={0}$'.format(n)))

for b in [2,3,5]:
        if gcd(n, b) == 1:
            aux = pow(b, n-1, n)
            if aux == 1:
                display(Latex('${3}^{0}\\equiv {1} \\mod {2}$. {2} SÍ es pseudoprimo en base {3}'.format(str({n-1}), aux, n, b)))
            else:
                display(Latex('${3}^{0}\\equiv {1} \\mod {2}$. {2} NO es pseudoprimo en base {3}'.format(str({n-1}), aux, n, b)))
            show(html('<br>'))

for _ in range(5):
        b = randint(1, 2**bits)
        while gcd(n, b) != 1:
            b = randint(1, 2**bits)
        aux = pow(b, n-1, n)
        if aux == 1:
            display(Latex('${3}^{0}\\equiv {1} \\mod {2}$. {2} SÍ es pseudoprimo en base {3}'.format(str({n-1}), aux, n, b)))
        else:
            display(Latex('${3}^{0}\\equiv {1} \\mod {2}$. {2} NO es pseudoprimo en base {3}'.format(str({n-1}), aux, n, b)))
        show(html('<br>'))



contar_bases_pp(n)
print()
contar_bases_pp(91)
print()
contar_bases_pp(561)


\(n=221\)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

221 NO es pseudoprimo en 176 bases

91 SÍ es pseudoprimo en 36 bases

91 NO es pseudoprimo en 36 bases

561 SÍ es pseudoprimo en 320 bases

561 NO es pseudoprimo en 0 bases

**Teorema.** si $n$ no pasa el test de Fermat en base $b$, $\text{mcd}(b,n)=1$, entonces no pasa el tests de Fermat para, como mínimo, la mitad de las bases posibles.

**Observaciones:**
* Excepto en el caso en que $n$ pase el test de Fermat en todas las bases, la probabilidad de que un entero compuesto pase el test de Fermat en una base elegida aleatoriamente es menor o igual que $\frac{1}{2}$.
* Números de Carmichael: enteros compuestos que pasan el test de Fermat en todas las bases:
    - 561 es el menor número de Carmichael.
    - Hay infinitos números de Charmichael.
    - $C(x)> x^{0.332}$ para $x$ grande. 


## <a href="#ippf" id="ppf">3. Pseudo primos fuertes. Test de Miller Rabin</a>

Sabemos que si $p$ es un primo impar entoces las únicas soluciones de $X^2\equiv 1 \mod p$ son $\pm 1$.

---

Sea $p$ primo impar, entonces podemos escribir $p-1=2^tp_0$ siendo $t\geq 1$ y $p_0$ impar. (*$t$ es el número de ceros finales en la representación binaria de $p-1$ y $p_0$ se obtiene a partir de $p-1$ eliminando los ceros finales.*)

Dado $b$, $b>0$, $\text{mcd}(b,p)=1$, definimos:
* $x_0= b^{p_0}\mod p$
* $x_i=x_{i-1}^2\mod p$, $i=1...t$.  (Notemos que $x_t\equiv b^{2^tp_0}\equiv 1 \mod p$.)

Por lo tanto:

* $x_t\equiv 1 \mod p$ (Notemos que $x_t\equiv b^{2^tp_0}\equiv 1 \mod p$.)
* $x_{t-1}\equiv \pm 1\mod p$
    - Si $x_{t-1}\equiv -1\mod p$, $-1\in\{x_0, x_1,...,x_{t-1}\}$
    - Si $x_{t-1}\equiv +1\mod p$
        * Si $x_{t-2}\equiv -1\mod p$, $-1\in\{x_0, x_1,...,x_{t-1}\}$
        * Si $x_{t-2}\equiv +1\mod p$
            * ...
                
Recapitulando:
 $-1\in\{x_0, x_1,...,x_{t-1}\}\quad \text{ó}\quad x_0\equiv 1 \mod p$ 
    
Dicho de otra manera: $\{x_0, x_1,...,x_{t}\}=\{...,-1,1,1,...,1\}$ o sea que o son todos 1 o delante del primer 1 hay un -1.

> ### Ejemplo

In [9]:
bits = 16
lista_enteros=[41, 561, 7*13, next_prime(randint(1, 2**bits)), 2*randint(1, 2**bits)+1, 2*randint(1, 2**bits)+1, 2*randint(1, 2**bits)+1, 2*randint(1, 2**bits)+1]
bases=[2, 3, 5, 10, 2*randint(1, 2**bits)+1, 2*randint(1, 2**bits)+1]

for n in lista_enteros:
    n1 = n-1
    t = 0
    while n1:
        if n1 & 1:
            break
        n1 >>= 1
        t += 1
    n0 = (n-1)/2**t

    show(html('<br> $n={0}, n-1={1}=2^{2}{3}$'.format(n, n-1, str({t}), n0)))

    for b in bases:
        x_i = [pow(b, n0, n)]
        for i in range(t):
            aux = pow(x_i[-1], 2, n)
            x_i.append(pow(x_i[-1], 2, n))
        y_i = [i if i != n-1 else -1 for i in x_i]
        display(Latex('Base {0}, {1}$\\equiv${2}'.format(b, x_i, y_i)))


<br> \(n=41, n-1=40=2^{3}5\)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

**Definición.** Diremos que $n$ entero positivo impar pasa el `test de Miller Rabin` en base $b$, $b>0$ y $\text{mcd}(b,n)=1$ si:

1. $x_t\equiv 1 \mod n$ (Fermat)
2. $-1\in\{x_0, x_1,...,x_{t-1}\}\quad \text{ó}\quad x_0\equiv 1 \mod p$ (esta condición implica la anterior).

siendo:
* $n-1=2^t n_0$, $n_0$ impar,
* $x_0= b^{n_0}\mod n$
* $x_i=x_{i-1}^2\mod n$, $i=1...t$. 

**Observaciones:**
* Si $n$ es primo, entonces pasa el test de Miller Rabin en cualquier base.
* Si $n$ no pasa el test de Miller Rabin en alguna base, entonces es compuesto.


**Definición.** Un $n$ entero compuesto positivo impar es `pseudoprimo fuerte` en base $b$ si pasa el test de Miller Rabin en dicha base.

**Observación:** Todo pseudoprimo fuerte en base $b$ también es pseudoprimo en la misma base.


> ### Ejemplo 

In [10]:
bits = 16
lista_enteros=[2*randint(1, 2**bits)+1 for _ in range(3)]
bases=[2, 3, 5, 97]

for n in lista_enteros:
    print(is_prime(n))
    n1 = n-1
    t = 0
    while n1:
        if n1 & 1:
            break
        n1 >>= 1
        t += 1
    n0 = (n-1)/2**t

    show(html('<br> $n={0}, n-1={1}=2^{2}{3}$'.format(n, n-1, str({t}), n0)))

    for b in bases:
        x_i = [pow(b, n0, n)]
        for i in range(t):
            aux = pow(x_i[-1], 2, n)
            x_i.append(pow(x_i[-1], 2, n))
        y_i = [i if i != n-1 else -1 for i in x_i]
        display(Latex('Base {0}, {1}$\\equiv${2}'.format(b, x_i, y_i)))



True


<br> \(n=27299, n-1=27298=2^{1}13649\)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<br> \(n=123885, n-1=123884=2^{2}30971\)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

**Teorema.** Un entero positivo compuesto impar $n$ es pseudoprimo fuerte en $\frac{\varphi(n)}{4}$ bases como máximo.

**Observaciones:**
* La probabilidad de que un entero positivo compuesto impar pase el test de MIller Rabin en una base elegida al azar es menor que $\frac{1}{4}$.
* La probabilidad de que un entero positivo compuesto impar pase el test de MIller Rabin $k$ bases elegida al azar es menor que $\frac{1}{4^k}$.
* El coste de aplicar el test de Miller Rabin a un entero $n$ es cúbico en el número de bits de $n$.

## <a href="#itest" id="test">4. Test de primalidad</a>

En la práctica, el proceso a seguir para buscar números, probablemente, primos es:

> 1. Generar $n$ impar del tamaño apropiado.
> 2. Se divide $n$ por primos menores que una cota $B$. (*Sólo el 20% de los candidatos pasa esta criba si B=256; añadir un primo más pasamos del 20.07% al 19.99%; si B=1000, pasan la criba el 16,19% de los candidatos.*)
3. Se comprueba si $n$ pasa el test de Miller Rabin en base 2. (*La mayoría de los enteros compuestos no lo pasan.*) 
4. Se aplica el test de Miller Rabin a $k$ bases tomadas al azar.

Si $n$ pasa (2)-(4) entonces es compuesto con probabilidad $<< \frac{1}{4^k}$.

Si $n$ falla en algún punto, se incrementa en 2 y se vuelve a (2). Si esta situación se repite en demasía, se vuelve a (1). 

**Observaciones:**
* La parte más costosa es el test de Miller Rabin que es $\mathcal O (\log^3{n})$.
* Las divisiones en (2) tienen un coste $\mathcal O (\log{n})$ ya que $B$ es independiente de $n$ y pequeño.
* Las probabilidades son $<< \frac{1}{4^k}$:
|  bits \ k      | 5 | 10     |
| :---        |    :----:   |          ---: |
| **250**   | $2^{-54}$       | $2^{-79}$   |
| **500**   | $2^{-85}$        | $2^{-119}$  |
* Si imponemos que la probabilidad sea menos que $2^{-80}$:
|  |  |  |  |  |
| - | :-: | :-: | :-: | :-: |
| **bits**   | 250      | 500   | 1000   | 2000   |
| **k**   | 12        | 6 | 3 | 2 |
* Existe un test determinístico *rápido* pero su coste es $\mathcal O (\log^6{n})$. (*Para un primo de 1000 bits supone pasar de tardar milisegundos a una semana.*)  
* Para enteros menores que $10^{50}$ es posible generar primos son total seguridad en tiempo real.

In [11]:
lista_primos = [i for i in range(256) if is_prime(i)]

def es_divisible(n, lista_primos):
    for i in lista_primos:
        if n%i == 0:
            return True
    return False

def Test_Miller_Rabin(n,b):
    if gcd(n,b) != 1:
        return None

    n1 = n-1
    t = 0
    while n1:
        if n1 & 1:
            break
        n1 >>= 1
        t += 1
    n0 = Integer((n-1)/2**t)

    x_i = Integer(pow(b, n0, n))
    if x_i == 1 or x_i == n-1:
        return True
    else:
        for _ in range(t):
            x_i = Integer(pow(x_i, 2, n))
            if x_i == n-1:
                return True
    return False


def Test_Miller_Rabin_bases(n, k = 10):
    for _ in range(k):
        base = randint(2, n-1)
        if gcd(base, n) != 1:
            return False
        if Test_Miller_Rabin(n, base) == False:
            return False
    return True


def Primo_probable(n_bits, lista_primos, k = 10):
    n = 2*randint(2**(n_bits-2), 2**(n_bits-1))+1
    aux = 0
    aux1 = 0
    while es_divisible(n, lista_primos) == True or Test_Miller_Rabin(n, 2) == False or Test_Miller_Rabin_bases(n, k = 10) == False:
        print('.', end='')
        n = n+2
        aux = aux+1
        if aux > n_bits//2:
            print('+', end='')
            n = 2*randint(2**(n_bits-2), 2**(n_bits-1))+1
            aux = 0
            aux1 += 1
    print(str(aux+aux1*n_bits//2)+':')
    return n


In [12]:
n = Primo_probable(2048, lista_primos)
print(is_prime(n), n)

....................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................580:
True 25032294367438551191489662947708738616937421893279075438671965576646413458898503497848236803133685403284060720665620209199809538171563710879498921828191505127662286381234283087573129926327320814003653794545782587689536848827438724422426066957685261733483426580148434821151728197400605465274688855548405049760663290275417362447793117257836832933438545941214756558087485674497211486270205919943031522233573495898

In [13]:
for _ in range(5):
    n = Primo_probable(1024,lista_primos)
    print(is_prime(n), n)
    print()

.......................................................................................................................................................................................................................215:
True 122146806767085108958007121983643332477754706982885540703871970148118850678436430337702935698675476341756376666678390702063952180756178487744527730172976197706542862784343546027223008232005719630898706784493974704724881750875595107381214987034018371805044490988833592756440078108398659451608227499545584544623

..............14:
True 90568834746788227896340788585602125727802170885558831309580076939329531174728301208454177217469071334979648845814243638216548023944427772039576079490660515442769174932831775382153146555603442291358406390457296770697801762467745568195326348331936977950041699262579079240510850472656106571629450197714021626731

...................................................................................................................................